# Records, Enums and Equality

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Model plain data with a record, and a fixed set of values with an enum.
- Tell == from equals, and write equals and hashCode so objects work as keys in sets and maps.

**In this notebook**, one idea at a time (run each cell before moving on):
1. Records: classes for plain data
2. Checking a record's values, and changing a copy
3. Enums: a fixed set of values
4. Enums with fields and methods
5. == or equals?
6. equals and hashCode in a class

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## 1. Records: classes for plain data

Many classes just hold data. A **record** writes all the usual code for you from one line: `record Song(String title, int seconds) {}` gives a constructor, the accessors `title()` and `seconds()`, a readable `toString`, and an `equals` and `hashCode` that compare the values. Its fields can't change after creation. You can still add methods inside the braces.

In [ ]:
record Song(String title, int seconds) {
    String length() {
        return seconds / 60 + ":" + String.format("%02d", seconds % 60);
    }
}

void main() {
    Song intro = new Song("Intro", 95);
    IO.println(intro);
    IO.println(intro.title() + " lasts " + intro.length());
}

🧪 **Try it:** add a field `String artist` to the record and fix the `new Song(...)` call. Then try `intro.seconds = 10;`: records can't change.

### ✏️ Exercise: An object's label

A Seat has a row letter and a number. Return its label, such as `A1`, from the record's accessors.

In [ ]:
record Seat(char row, int number) {}

String label(Seat seat) {
    // TODO: join seat.row() and seat.number()
    return "";
}

void main() {
    check("A1", label(new Seat('A', 1)), "A1");
    check("C12", label(new Seat('C', 12)), "C12");
}

<details>
<summary>Show an answer</summary>

```java
record Seat(char row, int number) {}

String label(Seat seat) {
    return "" + seat.row() + seat.number();
}

void main() {
    check("A1", label(new Seat('A', 1)), "A1");
    check("C12", label(new Seat('C', 12)), "C12");
}
```

</details>

## 2. Checking a record's values, and changing a copy

A record can check its values in a **compact constructor**: the record's name with no parentheses, which runs before the fields are set. Since a record never changes, "changing" one means creating a new record with one value replaced, usually in a method named `withSomething`.

In [ ]:
record Song(String title, int seconds) {
    Song {                                        // the compact constructor
        if (seconds <= 0) {
            throw new IllegalArgumentException("a song needs a positive length");
        }
    }

    Song withSeconds(int newSeconds) {
        return new Song(title, newSeconds);       // a new record; this one stays
    }
}

void main() {
    Song a = new Song("Intro", 95);
    Song b = a.withSeconds(120);
    IO.println(a + " / " + b);
    try {
        new Song("Silence", 0);
    } catch (IllegalArgumentException e) {
        IO.println("Refused: " + e.getMessage());
    }
}

🧪 **Try it:** add a `withTitle` method. Then also refuse an empty title in the compact constructor.

## 3. Enums: a fixed set of values

When a value can only be one of a few options (the days of the week, the sizes of a cup), an **enum** lists them. The compiler then refuses anything else, `values()` gives all of them in order, `valueOf("SMALL")` turns text into one, and a `switch` can go through every option without a `default`.

In [ ]:
enum Direction { NORTH, EAST, SOUTH, WEST }

Direction turnRight(Direction d) {
    return switch (d) {
        case NORTH -> Direction.EAST;
        case EAST -> Direction.SOUTH;
        case SOUTH -> Direction.WEST;
        case WEST -> Direction.NORTH;
    };
}

void main() {
    Direction facing = Direction.valueOf("NORTH");
    facing = turnRight(facing);
    IO.println("Now facing " + facing + " (number " + facing.ordinal() + ")");
    for (Direction d : Direction.values()) {
        IO.print(d + " ");
    }
    IO.println();
}

🧪 **Try it:** delete one `case` and read the error. Then try `Direction.valueOf("UP")` and read the exception.

### ✏️ Exercise: Cup sizes

Return how many millilitres each size holds: SMALL 250, MEDIUM 350, LARGE 500. Use a switch over the enum.

In [ ]:
enum Size { SMALL, MEDIUM, LARGE }

int ml(Size size) {
    // TODO: a switch with one case per size
    return 0;
}

void main() {
    check("small", ml(Size.SMALL), 250);
    check("medium", ml(Size.MEDIUM), 350);
    check("large", ml(Size.LARGE), 500);
}

<details>
<summary>Show an answer</summary>

```java
enum Size { SMALL, MEDIUM, LARGE }

int ml(Size size) {
    return switch (size) {
        case SMALL -> 250;
        case MEDIUM -> 350;
        case LARGE -> 500;
    };
}

void main() {
    check("small", ml(Size.SMALL), 250);
    check("medium", ml(Size.MEDIUM), 350);
    check("large", ml(Size.LARGE), 500);
}
```

</details>

## 4. Enums with fields and methods

Each value of an enum is an object, so an enum can have fields, a constructor and methods, like a class. The values pass their data to the constructor in parentheses. That keeps each value's data next to its name, instead of in a separate switch.

In [ ]:
enum Planet {
    MERCURY(0.38), EARTH(1.0), JUPITER(2.53);

    private final double gravity;     // relative to Earth

    Planet(double gravity) {
        this.gravity = gravity;
    }

    double weigh(double kgOnEarth) {
        return kgOnEarth * gravity;
    }
}

void main() {
    for (Planet p : Planet.values()) {
        IO.println(p + ": " + String.format("%.1f", p.weigh(60)) + " kg");
    }
}

🧪 **Try it:** add MARS with 0.38 gravity. Then add a field with each planet's position from the Sun, and print it.

## 5. == or equals?

For objects, `==` asks "**the same object**?": two `new`s make two objects, even with equal values. `equals` asks "**the same values**?", but only if the class says how to compare them. Records and Strings already do; for enums, `==` is fine, because each value exists only once. The slides show it in memory.

#### 🎞️ Objects, references and equality

![Two news make two objects even with equal values; assigning shares one object; == compares objects and equals compares values](objects-memory-steps.svg)

In [ ]:
record Seat(char row, int number) {}

void main() {
    Seat s1 = new Seat('A', 1);
    Seat s2 = new Seat('A', 1);
    Seat s3 = s1;
    IO.println(s1 == s2);
    IO.println(s1.equals(s2));
    IO.println(s1 == s3);
}

🧪 **Try it:** predict the three lines before running. Then change `record Seat` to a plain class with the same fields and a constructor: what does `s1.equals(s2)` give now?

## 6. equals and hashCode in a class

A plain class inherits an `equals` that only compares references. To compare values, override `equals(Object other)`: check that `other` is the same kind of object, then compare the fields. Override `hashCode` **too**, with `Objects.hash(...)` of the same fields: a `HashSet` or `HashMap` first looks at the hash, and two equal objects with different hashes are treated as different.

In [ ]:
class Money {
    private final long cents;
    private final String currency;

    Money(long cents, String currency) {
        this.cents = cents;
        this.currency = currency;
    }

    @Override
    public boolean equals(Object other) {
        return other instanceof Money m && cents == m.cents && currency.equals(m.currency);
    }

    @Override
    public int hashCode() {
        return Objects.hash(cents, currency);
    }
}

void main() {
    Money a = new Money(500, "BRL");
    Money b = new Money(500, "BRL");
    IO.println(a.equals(b));
    Set<Money> wallet = new HashSet<>(List.of(a, b));
    IO.println(wallet.size());
}

🧪 **Try it:** delete the `hashCode` method and run: `equals` still says true, but the set may keep both. Delete `equals` too and compare.

## In short

A **record** is a class for plain data in one line: `record Song(String title, int seconds) {}` writes the constructor, the accessors `title()` and `seconds()`, `toString`, `equals` and `hashCode`, and its fields never change. An **enum** lists a fixed set of values, such as `enum Size { SMALL, MEDIUM, LARGE }`.

```java
record Song(String title, int seconds) {}

enum Size { SMALL, MEDIUM, LARGE }

void main() {
    Song a = new Song("Intro", 95);
    Song b = new Song("Intro", 95);
    IO.println(a == b);          // false: two objects
    IO.println(a.equals(b));     // true: the same values
    Set<Song> playlist = new HashSet<>(List.of(a, b));
    IO.println(playlist.size()); // 1: equal songs count once

Size s = Size.valueOf("MEDIUM");
    IO.println(s + " is size number " + (s.ordinal() + 1));
}
```

`==` asks "the same object?"; `equals` asks "the same values?". A plain class inherits an `equals` that only compares references, so to compare values you override `equals` **and** `hashCode` together.

**Watch out:** two objects that are `equals` must have the same `hashCode`, or a `HashSet` or `HashMap` treats them as different.

## Readings

- [Object as a Superclass](https://dev.java/learn/language/oop/inheritance/objects/) (dev.java (Oracle), 8 min): The sections on toString, equals and hashCode: why equals must be overridden to compare values, and why hashCode must change along with it. You can skip clone and finalize.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Points as Records** · Easy
- **Coins (enum)** · Medium
- **Equal Points (equals and hashCode)** · Hard
- Test: **Library System** · Hard